## Optional: Hyperparameter optimization (HPO)
<div class="alert alert-info">This section is optional – you don't need it for the further course of the workshop. You can move to the <b>Clean-up</b> section.</div>

[Amazon SageMaker automatic model tuning](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html), also called hyperparameter optimization (HPO), finds the best performing model against a defined objective metric. In V3, `HyperparameterTuner` takes a `ModelTrainer` (via `model_trainer` parameter) instead of a V2 `Estimator`.

In [ ]:
suffix = strftime('%d-%H-%M-%S', gmtime())
hpo_experiment_name = f"from-idea-to-prod-hpo-{suffix}"
registered_hpo_model_name = f"from-idea-to-prod-hpo-model-{suffix}"

mlflow.set_experiment(hpo_experiment_name)

In [ ]:
# import required HPO objects
from sagemaker.train import HyperparameterTuner, ModelTrainer
from sagemaker.train.tuner import (
    CategoricalParameter,
    ContinuousParameter,
    IntegerParameter,
)

In [ ]:
from sagemaker.train.configs import SourceCode, Compute, InputData

# set up hyperparameter ranges
hp_ranges = {
    'min_child_weight': ContinuousParameter(1, 10),
    'max_depth': IntegerParameter(1, 10),
    'alpha': ContinuousParameter(0, 5),
    'eta': ContinuousParameter(0, 1),
    'colsample_bytree': ContinuousParameter(0, 1),
}

# set up the objective metric
objective = 'validation:auc'

# set up the ModelTrainer for HPO
hpo_trainer = ModelTrainer(
    training_image=xgb_training_image,
    source_code=SourceCode(
        source_dir='./training',
        entry_script='train.py',
        requirements='requirements.txt',
    ),
    compute=Compute(instance_type=train_instance_type, instance_count=train_instance_count),
    hyperparameters=hyperparams,
    role=sm_role,
    base_job_name='from-idea-to-prod-training',
    output_data_config={'s3_output_path': output_s3_url},
    environment={
        'MLFLOW_TRACKING_ARN': mlflow_arn,
        'MLFLOW_EXPERIMENT_NAME': hpo_experiment_name,
        'USER': user_profile_name,
        'REGION': region,
    },
)

# instantiate a HPO object
tuner = HyperparameterTuner(
    model_trainer=hpo_trainer,
    objective_metric_name=objective,
    hyperparameter_ranges=hp_ranges,
    max_jobs=10,
    max_parallel_jobs=3,
    strategy='Bayesian',
    objective_type='Maximize',
    base_tuning_job_name='from-idea-to-prod-hpo',
    early_stopping_type='Auto',
)

### Run the HPO
Now run the HPO job. It takes about 10 minutes to complete.

In [ ]:
tuner.fit(
    input_data_config=[
        InputData(channel_name='train', data_source=train_s3_url),
        InputData(channel_name='validation', data_source=validation_s3_url),
    ],
)

In [ ]:
tuner.describe()['HyperParameterTuningJobStatus']

To see the details on the HPO job in the SageMaker console click on the link constructed by the cell below:

In [ ]:
# Show the HPO job
display(
    HTML('<b>See the SageMaker <a target="top" href="https://{}.console.aws.amazon.com/sagemaker/home?region={}#/hyper-tuning-jobs/{}">HPO job</a></b>'.format(
            region, region, tuner.latest_tuning_job.job_name))
)

### See HPO results
Now get the results and see the best training job.

In [ ]:
best_training_job = tuner.describe()['BestTrainingJob']

In [ ]:
best_training_job

### Register the best HPO model in the model registry
Register the best HPO model with the hyperparameters, metrics, and model artifacts in the MLflow model registry.

In [ ]:
# find the run with the best HPO model
best_hpo_model_run_id = mlflow.search_runs(
    experiment_ids=[mlflow.get_experiment_by_name(hpo_experiment_name).experiment_id], 
    filter_string=f"tags.mlflow.source.name LIKE '%{best_training_job['TrainingJobArn'].split('/')[-1]}%'",
)['run_id'][0]

In [ ]:
best_hpo_model_run_id

In [ ]:
# Register the model
model_uri = f"runs:/{best_hpo_model_run_id}/model"
registered_hpo_model_version = mlflow.register_model(model_uri, registered_hpo_model_name)

As the next step you need to deploy the best HPO model and test it. You can choose one of the following options below.

### Run local inference with the best HPO model

In [ ]:
# get the model from the MLflow model registry
model_uri = registered_hpo_model_version.source
model = mlflow.xgboost.load_model(model_uri)

# load data
test_x = loadtxt("tmp/test_x.csv", delimiter=",")
test_y = loadtxt("tmp/test_y.csv", delimiter=",")

# predict
dtest = xgb.DMatrix(test_x)

predictions = np.array(model.predict(dtest), dtype=float).squeeze()
predictions

In [ ]:
pd.crosstab(
    index=test_y,
    columns=np.round(predictions), 
    rownames=['actuals'], 
    colnames=['predictions']
)

In [ ]:
test_auc = roc_auc_score(test_y, predictions)
print(f"Test-auc: {test_auc:.4f}")

There is only a small improvements for the model metrics. It can indicate, that the XGBoost model is already at it's limit. You might want to explore other model types to improve the prediction accuracy for this use case.